In [ ]:
import pandas as pd, numpy as np
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
import pathlib

In [ ]:
location = "India"
vehicle = "rice"

In [ ]:
location = location.title()

In [ ]:
pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

In [ ]:
asfr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
).value
asfr[asfr > 0]

In [ ]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

In [ ]:
births = pop * asfr
births[births > 0]

In [ ]:
births = births.sum()
f"{int(births):,}"

In [ ]:
sim_baseline_births = pd.read_parquet(
    f"../../0200_pregnancy_sim/sim_results/{vehicle}/{location.lower()}/pregnancy_outcome_count.parquet"
)
sim_baseline_births

In [ ]:
sim_baseline_births = sim_baseline_births[
    (sim_baseline_births.scenario == "baseline")
    & (sim_baseline_births.sub_entity == "live_birth")
]
sim_baseline_births

In [ ]:
n_random_seeds = sim_baseline_births.random_seed.nunique()
sim_baseline_births = sim_baseline_births.groupby("input_draw").value.sum().mean()
sim_baseline_births

In [ ]:
births

In [ ]:
scalar = births / sim_baseline_births
scalar

In [ ]:
# `scalar` multiplies every child number this notebook writes, and until now nothing
# bounded it -- it was displayed and then applied. A zero simulated denominator makes it
# `inf`, which propagates silently into every rescaled parquet rather than failing.
assert np.isfinite(scalar) and scalar > 0, (
    f"rescaling factor is not a positive finite number: {scalar}. "
    f"GBD births={births:,.0f}, simulated baseline live births={sim_baseline_births:,.0f} "
    f"over {n_random_seeds} seeds. Every parquet below is multiplied by this."
)

# The scalar itself cannot be banded: it scales as 1/n_random_seeds, so it was 1.06-3.10
# for the published 200-seed run and 32.5-48.1 for a 10-seed one. What *is* seed
# independent is the simulated cohort size per seed, measured at 26k-44k live births
# across both GBD 2021 and GBD 2023. The band below leaves ~25x margin either side; its
# job is to catch a collapsed or missing denominator, not to pin epidemiology.
births_per_seed = sim_baseline_births / n_random_seeds
assert 1_000 <= births_per_seed <= 1_000_000, (
    f"{births_per_seed:,.0f} simulated live births per seed is outside the plausible "
    f"1,000-1,000,000 range ({sim_baseline_births:,.0f} over {n_random_seeds} seeds). "
    "Either the simulation produced almost nothing, or pregnancy_outcome_count.parquet "
    "is not what this notebook expects."
)
f"scalar={scalar:,.4f} from {births_per_seed:,.0f} live births/seed x {n_random_seeds} seeds"

In [ ]:
# NOTE: 'person_time' was renamed to 'person_time_population' when PersonTimeObserver
# became a PublicHealthObserver, so the automated V&V loader can discover it.
for result in ["ylds", "ylls", "deaths", "person_time_population"]:
    df = pd.read_parquet(
        f"../../0300_child_sim/sim_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    df.value *= scalar
    path = pathlib.Path(
        f"../results/rescaled_child_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)